# Módulo 3 de Analítica de Datos — Delta Lake

En este módulo vamos a trabajar con **Delta Lake**, el formato de almacenamiento por defecto en Databricks y la base del **Lakehouse**.

### ¿Qué es Delta Lake?
Delta Lake es un **formato de almacenamiento abierto** que extiende Apache Parquet con un **registro de transacciones** (`_delta_log/`). Sobre ficheros Parquet, añade:

- **Transacciones ACID**: lecturas y escrituras consistentes incluso con múltiples productores/consumidores concurrentes.
- **Schema enforcement**: rechaza escrituras que no cumplen el esquema (evita corromper la tabla).
- **Schema evolution**: permite añadir columnas de forma controlada (`mergeSchema`, `overwriteSchema`).
- **Time travel**: poder consultar el estado de la tabla en una versión o instante anteriores (`VERSION AS OF`, `TIMESTAMP AS OF`).
- **Operaciones DML**: `UPDATE`, `DELETE`, `MERGE INTO` (upserts), que en CSV/Parquet puro no existen.
- **Optimizaciones**: `OPTIMIZE` (compactación de ficheros pequeños), `ZORDER BY` (clustering por columnas), *liquid clustering*, *data skipping*, *predicate pushdown*.

### CSV/Parquet vs Delta
| Característica | CSV / Parquet | Delta |
|---|---|---|
| Lectura paralela | Sí | Sí |
| Tipos y compresión | Solo Parquet | Sí (basado en Parquet) |
| Transacciones ACID | No | Sí |
| `UPDATE` / `DELETE` / `MERGE` | No | Sí |
| Time travel | No | Sí |
| Lineage en Unity Catalog | Limitado | Completo |

### Tabla **managed** vs **external**
- **Managed table**: la gestiona Unity Catalog. Si haces `DROP TABLE`, **se borran los datos**.
- **External table**: apunta a una ruta concreta (`LOCATION ...`). Al hacer `DROP TABLE`, los datos permanecen.

Veremos las dos formas en este notebook.

## 1. Carga de fichero CSV en un Volume de Unity Catalog

Como en el módulo anterior, **los Volumes** son la ubicación gobernada para ficheros no tabulares (CSV, JSON, imágenes, modelos…). Vivirán en una jerarquía `catalog → schema → volume → archivo`.

### `CREATE VOLUME IF NOT EXISTS`
- `IF NOT EXISTS` hace la sentencia **idempotente**: no falla si el volumen ya existía.
- Sin especificar catálogo/schema, se crea en los **actuales** (`current_catalog()` / `current_schema()`).
- Para volúmenes externos: `CREATE EXTERNAL VOLUME <name> LOCATION '<cloud-path>'`.

> **Permisos opcionales**: para que otros usuarios puedan leer/escribir en el volumen necesitarán `READ VOLUME` y/o `WRITE VOLUME`:
> ```sql
> GRANT READ VOLUME ON VOLUME <catalog>.<schema>.spark_lab TO `<usuario_o_grupo>`;
> ```

In [ ]:
%sql
-- Creamos un volumen 'spark_lab' en el catálogo y schema actuales si no existe.
CREATE VOLUME IF NOT EXISTS spark_lab

### Descarga del fichero `products.csv`

Descargamos un fichero de ejemplo con datos de productos desde un repositorio público de GitHub y lo dejamos en el Volume.

Detalles a tener en cuenta:
- `spark.sql("SELECT current_catalog()")` resuelve el catálogo activo en tiempo de ejecución, lo que hace el notebook **portable** (funciona igual en dev, test y prod).
- Las rutas de Volumes siguen el formato POSIX `/Volumes/<catalog>/<schema>/<volume>/...` y son accesibles desde Python (`open`), Spark (`spark.read`) y la línea de comandos (`%sh`).
- `response.raise_for_status()` aborta pronto si la URL devuelve un error HTTP.

In [ ]:
import requests

# Resolvemos dinámicamente el catálogo actual.
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Ruta base del Volume usando el formato /Volumes/<catalog>/<schema>/<volume>.
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# En este módulo solo necesitamos un fichero (catálogo de productos).
files = ["products.csv"]

# Descargamos cada fichero y lo guardamos en el Volume.
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()  # Aborta si hay error HTTP (4xx/5xx).

    # 'wb' = write binary; response.content devuelve bytes.
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

## 2. Lectura del CSV en un DataFrame

Leemos el fichero usando `spark.read.load(...)` con `format='csv'` y `header=True` para que la primera fila se interprete como cabecera (nombres de columnas en lugar de `_c0`, `_c1`, ...).

### Opciones útiles al leer CSV
- `header=True` → primera fila = nombres de columna.
- `inferSchema=True` → Spark deduce los tipos (hace una pasada extra; en producción es mejor un esquema explícito).
- `sep=","` → separador (alias `delimiter`).
- `nullValue="NA"` → texto que se interpretará como `NULL`.
- `mode="PERMISSIVE" | "DROPMALFORMED" | "FAILFAST"` → comportamiento ante filas mal formadas.
- `multiLine=True` → un campo puede contener saltos de línea.

Sin `inferSchema`, todas las columnas se leerán como `string`. Para este ejemplo es suficiente: a continuación lo convertiremos a Delta y, llegado el caso, podríamos castear los tipos antes.

In [ ]:
# header=True: la primera fila contiene los nombres de columna.
# Sin inferSchema, todas las columnas serán de tipo string.
df = spark.read.load(
    f'/Volumes/{catalog_name}/default/spark_lab/products.csv',
    format='csv',
    header=True
)

# Limitamos a 10 filas para una vista previa rápida.
display(df.limit(10))

## 3. Conversión a tabla Delta

Vamos a guardar el DataFrame como **Delta**. Existen dos enfoques principales:

1. **Path-based table** (sección siguiente): escribimos en una **ruta concreta** del Volume con `df.write.format("delta").save(path)`. La tabla "vive" en esa carpeta y se referencia por su ruta. Es lo más parecido a una *external table* pero sin registrarla en el catálogo.
2. **Managed table en el catálogo** (al final del notebook): la registramos con `saveAsTable("<schema>.<nombre>")`. Unity Catalog gestiona su ubicación y se accede por su nombre lógico.

### Modos de escritura (`mode`)
- `"errorifexists"` (por defecto): falla si ya existe.
- `"append"`: añade datos al final (incremental).
- `"overwrite"`: reemplaza el contenido (puede combinarse con `option("overwriteSchema", "true")`).
- `"ignore"`: si existe, no hace nada.

### Opciones de escritura típicas
- `.partitionBy("col")` → particiona por una columna (útil con cardinalidades bajas).
- `.option("mergeSchema", "true")` → permite añadir columnas nuevas en un append.
- `.option("overwriteSchema", "true")` → reemplaza el esquema en un overwrite.

> **Consideración opcional**: en Delta moderno se prefiere **liquid clustering** (`CLUSTER BY`) sobre `partitionBy` para la mayoría de casos: es más flexible y se adapta sin reescribir la tabla.

### 3.1 Almacenamiento path-based en el Volume

Escribimos los datos como Delta en una subcarpeta del Volume. Internamente, Delta crea:

```
products-delta/
├── _delta_log/                ← registro de transacciones (JSON + checkpoints Parquet)
│   ├── 00000000000000000000.json
│   └── ...
├── part-00000-...-c000.snappy.parquet  ← datos en Parquet
└── part-00001-...-c000.snappy.parquet
```

Ese `_delta_log/` es el corazón del formato: cada operación (insert, update, delete, optimize) se registra como una versión nueva, lo que habilita ACID y *time travel*.

In [ ]:
# Ruta donde vivirá la tabla Delta. Es solo una carpeta dentro del Volume.
delta_table_path = f"/Volumes/{catalog_name}/default/spark_lab/delta/products-delta"

# Escribimos el DataFrame como Delta. Esto crea ficheros Parquet + carpeta _delta_log/.
# Al ser la primera escritura, el modo por defecto 'errorifexists' es válido.
df.write.format("delta").save(delta_table_path)

### 3.2 Manipular la tabla Delta con la API `DeltaTable`

El módulo `delta.tables` ofrece una API programática para realizar operaciones DML sobre tablas Delta sin escribir SQL:

- `DeltaTable.forPath(spark, path)` → carga la tabla por ruta.
- `DeltaTable.forName(spark, "<catalog>.<schema>.<tabla>")` → carga por nombre del catálogo.
- `.update(condition, set)` → equivalente a `UPDATE ... SET ... WHERE ...`.
- `.delete(condition)` → equivalente a `DELETE FROM ... WHERE ...`.
- `.merge(source, condition).whenMatchedUpdate(...).whenNotMatchedInsert(...).execute()` → upserts (`MERGE INTO`).
- `.history(n)` → últimas `n` operaciones.
- `.toDF()` → devuelve un DataFrame con el contenido actual.
- `.vacuum(retentionHours)` → borra ficheros antiguos no referenciados (¡cuidado: limita el time travel!).

### Sobre los argumentos de `update`
- `condition` es una **expresión SQL** (string) o un objeto `Column` de PySpark.
- `set` es un **diccionario** `{"columna": expresión_string_o_Column}`.
- Las expresiones del lado derecho se evalúan **por fila** (`"ListPrice * 0.9"` toma el valor actual de cada fila y lo multiplica).

Equivalente en SQL:
```sql
UPDATE delta.`<delta_table_path>`
SET ListPrice = ListPrice * 0.9
WHERE ProductID = 771;
```

> **Nota**: una operación `UPDATE` en Delta no edita Parquets en sitio. Lee los ficheros afectados, **reescribe** versiones nuevas con los cambios y registra la operación en `_delta_log/`. Por eso es transaccional y se puede deshacer con time travel.

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

# Creamos el objeto DeltaTable a partir de la ruta donde guardamos los datos.
deltaTable = DeltaTable.forPath(spark, delta_table_path)

# Aplicamos un UPDATE: rebajamos un 10% el precio del producto con ProductID = 771.
# - condition: filtro WHERE (string SQL).
# - set: diccionario con la asignación; el valor también es una expresión SQL.
deltaTable.update(
   condition = "ProductID == 771",
   set = { "ListPrice": "ListPrice * 0.9" }
)

# toDF() devuelve un DataFrame con el estado actual de la tabla; mostramos 10 filas.
deltaTable.toDF().show(10)

## 4. Releer la tabla Delta como un DataFrame nuevo

Cualquier consumidor (otro notebook, un job, una herramienta de BI) puede leer la tabla apuntando a su ruta con `format("delta")`. Cada lectura ve la **última versión confirmada** del registro de transacciones, lo que garantiza consistencia incluso con escrituras concurrentes.

### Time travel (lectura de versiones anteriores)
Delta permite leer versiones pasadas de la tabla sin restaurarla. Es muy útil para auditar cambios o reproducir un análisis:

```python
# Por número de versión
spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)

# Por timestamp
spark.read.format("delta").option("timestampAsOf", "2026-05-23").load(delta_table_path)
```

Equivalente en SQL:
```sql
SELECT * FROM delta.`<delta_table_path>` VERSION AS OF 0;
SELECT * FROM delta.`<delta_table_path>` TIMESTAMP AS OF '2026-05-23';
```

> El time travel está limitado por la retención del log y por `VACUUM`: por defecto, Delta retiene 30 días de log y 7 días de ficheros de datos eliminados. Ajustables con `delta.logRetentionDuration` y `delta.deletedFileRetentionDuration`.

In [ ]:
# Lectura estándar de la tabla Delta por ruta.
# Lee la versión más reciente confirmada en _delta_log/.
new_df = spark.read.format("delta").load(delta_table_path)
new_df.show(10)

## 5. Histórico de operaciones (`history`)

`deltaTable.history(n)` devuelve un DataFrame con las **últimas `n` operaciones** efectuadas sobre la tabla. Cada fila representa una versión y trae metadatos muy útiles:

| Columna | Descripción |
|---|---|
| `version` | Número de versión (incremental) |
| `timestamp` | Cuándo ocurrió |
| `userId` / `userName` | Quién la ejecutó |
| `operation` | `WRITE`, `UPDATE`, `DELETE`, `MERGE`, `OPTIMIZE`, ... |
| `operationParameters` | Detalles (predicados, modo de escritura...) |
| `operationMetrics` | Filas/ficheros afectados |
| `readVersion` | Versión que leyó la operación |
| `isolationLevel` | Nivel de aislamiento utilizado |

### Argumentos de `show(...)` aquí utilizados
`deltaTable.history(10).show(10, False, True)` equivale a:
- `numRows=10` → muestra 10 filas.
- `truncate=False` → no recorta los valores largos.
- `vertical=True` → imprime cada fila en formato vertical (cada columna en su línea), mucho más legible cuando hay muchas columnas.

Equivalente en SQL:
```sql
DESCRIBE HISTORY delta.`<delta_table_path>`;
```

In [ ]:
# Mostramos las últimas 10 operaciones de la tabla en formato vertical (más legible).
# Parámetros de show(numRows, truncate, vertical):
#   numRows=10  -> 10 filas
#   truncate=False -> no recorta cadenas largas
#   vertical=True  -> imprime una columna por línea
deltaTable.history(10).show(10, False, True)

## 6. Crear una tabla en el Data Catalog (managed table)

Hasta ahora hemos trabajado por **ruta** (path-based). Si queremos exponer la tabla a herramientas SQL, BI o a otros usuarios usando un nombre lógico, debemos **registrarla en Unity Catalog** con `saveAsTable`.

### `saveAsTable("<schema>.<tabla>")`
- Si **no especificas** `LOCATION`, se crea como **managed table**: Unity Catalog elige la ubicación física y la borra al hacer `DROP`.
- Si especificas `LOCATION` (con `option("path", "...")`), se crea como **external table**.
- El nombre puede ser de tres niveles: `<catalog>.<schema>.<tabla>` (recomendado en Unity Catalog) o de dos `<schema>.<tabla>` (usa el catálogo activo).

### Diferencias clave: managed vs external
| Aspecto | Managed | External |
|---|---|---|
| Ubicación física | La gestiona UC | La eliges tú |
| `DROP TABLE` | Borra datos | **No** borra datos |
| Uso típico | Tablas internas | Datos compartidos con sistemas externos |

### Modos al registrar
Igual que `save`: `mode("overwrite")`, `mode("append")`, etc. Si la tabla ya existía, hay que tener cuidado con sobrescrituras.

Equivalente SQL:
```sql
CREATE TABLE default.ProductsManaged USING DELTA AS
SELECT * FROM <origen>;
```

In [ ]:
# Registramos el DataFrame original como tabla managed Delta en el schema 'default'.
# Como no indicamos LOCATION, Unity Catalog gestiona el almacenamiento.
# Acceso posterior: SELECT * FROM default.ProductsManaged;
df.write.format("delta").saveAsTable("default.ProductsManaged")

## 7. Consultar la tabla con SQL

Una vez registrada, la tabla es accesible mediante **Spark SQL** (`%sql` o `spark.sql(...)`), desde un **SQL Warehouse**, desde herramientas de BI conectadas (Power BI, Tableau...) o vía JDBC/ODBC.

### `USE` y nombres calificados
- `USE default;` cambia el **schema activo** para la sesión SQL, así puedes referirte a la tabla sin prefijo.
- También puedes saltarte el `USE` y usar el nombre completo: `SELECT * FROM default.ProductsManaged;` o `SELECT * FROM <catalog>.default.ProductsManaged;`.

### Magic command `%sql`
El prefijo `%sql` al principio de la celda indica a Databricks que la celda contiene SQL en lugar de Python. Internamente equivale a llamar a `spark.sql(...)` con cada sentencia.

> **Consideración opcional**: para un análisis ad-hoc puede ser cómodo usar `display(spark.sql("SELECT ..."))` desde Python — así puedes seguir transformando el resultado con la API de DataFrames y, además, generar gráficos interactivos con `display()`.

In [ ]:
%sql
-- Cambiamos el schema activo de la sesión SQL.
USE default;
-- Consultamos todos los registros de la tabla managed que acabamos de crear.
SELECT * FROM ProductsManaged;

## Resumen del módulo

En este notebook hemos cubierto los fundamentos de **Delta Lake** en Databricks:

1. **Ingesta**: descargar un CSV y dejarlo en un Volume de Unity Catalog.
2. **Lectura**: cargarlo como DataFrame con `spark.read.load(..., header=True)`.
3. **Conversión a Delta** (path-based): `df.write.format("delta").save(path)` y la estructura `_delta_log/` + Parquet que se crea por debajo.
4. **API `DeltaTable`**: `forPath`, `update`, `delete`, `merge`, `toDF`, `history`, `vacuum`.
5. **Time travel**: `versionAsOf` / `timestampAsOf`, y los retention thresholds que lo limitan.
6. **Histórico de operaciones**: cómo auditar quién hizo qué y cuándo.
7. **Registro en Unity Catalog**: `saveAsTable` para crear una **managed table** y consultarla con SQL.

### Próximos pasos sugeridos
- Probar **`MERGE INTO`** para upserts incrementales (patrón típico de SCD tipo 1/2).
- Aplicar **`OPTIMIZE`** y **`ZORDER BY`** (o **liquid clustering**) para acelerar consultas.
- Configurar **change data feed** (`delta.enableChangeDataFeed = true`) para emitir eventos de cambios y construir pipelines incrementales.
- Explorar **Delta Live Tables (DLT)** / **Lakeflow Declarative Pipelines** para definir pipelines declarativos sobre Delta.
- Revisar la pestaña **Lineage** de Unity Catalog para ver cómo se conectan tus notebooks, jobs, tablas y dashboards.